# 00 Part 1 — Data Preparation (Women Shoes Size 8, Mar 2025 – Mar 2026)

**Input :** `demand_modeling_data_8_v2/amazon_shoes_8_combined.parquet`

**Output:**
```
demand_modeling_data_women_8_v2/
  code/
    main_train_keys.csv          ← place next to notebooks 01 and 04
    main_val_keys.csv
  data/amzn_shoes_monthly_diffs_ffill_fixed_splits/
    train-00000-of-00001.parquet
    validation-00000-of-00001.parquet
```

**Pipeline order (senior DS standard):**
1. Load → remove dirty ASINs → filter Women
2. Drop dead columns → fix dtypes → rebuild subcat
3. Forward fill on FULL data (Jan–Mar) — Jan/Feb acts as seed for March
4. Create NaN flags on trimmed window only
5. Trim to Mar 2025 onward
6. Drop incomplete ASINs (entire ASIN, not rows) — show list first
7. Final checks → split → save → removal log

## ① Setup

In [ ]:
# Dependencies: pyarrow, pandas (install locally)
print('Local mode')

## ② Config

In [ ]:
import os, shutil
import pandas as pd
import numpy as np
import random
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

ROOT = str(PROJECT_ROOT) + '/'
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE = str(PROJECT_ROOT.parent / 'demand_modeling_data_8_v2' / 'amazon_shoes_8_combined.parquet')
KEY_DIR  = ROOT + 'code/'
DATA_DIR = ROOT + 'data/amzn_shoes_monthly_diffs_ffill_fixed_splits/'

os.makedirs(KEY_DIR,  exist_ok=True)
os.makedirs(ROOT + "output/", exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ── Window ──────────────────────────────────────────────────────────────────
START_DATE = '2025-03-01'   # Analysis window start — Jan/Feb used as fill seed only
END_DATE   = '2026-03-31'

# ── Split ───────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.5
SEED        = 42

# ── Columns ─────────────────────────────────────────────────────────────────
FFILL_COLS = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

DROP_COLS = [
    'SALES_RANK_original', 'PRICE_original', 'BUYBOX_PRICE_original',
    'gender', 'manufacturer', 'brand', 'model', 'color', 'size',
]

REQUIRED_COMPLETE = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

NAN_TOLERANCE = 0.0

PAPER_SCHEMA = {
    'ASIN': 'object', 'window': 'float64', 'date': 'datetime64[ns]',
    'SALES_RANK': 'float64', 'PRICE': 'float64', 'BUYBOX_PRICE': 'float64',
    'text': 'object', 'RATING': 'float64', 'REVIEW_COUNT': 'float64',
    'subcat': 'object', 'subcat_aggregated': 'object',
    'New Offer Count: Current': 'int64',
    'Count of retrieved live offers: New, FBA': 'int64',
    'Count of retrieved live offers: New, FBM': 'int64',
    'Lightning Deals: Upcoming Deal': 'int64',
    'Buy Box: Is FBA': 'int64',
    'image': 'object',
}

removal_log = []
def log_removal(asins, reason):
    for asin in asins:
        removal_log.append({'ASIN': asin, 'reason': reason})

print('✅ Config ready')
print(f'  Input       : {INPUT_FILE}')
print(f'  Output root : {ROOT}')
print(f'  Key CSVs    : {KEY_DIR}  (placed next to 01_1, 01_2, 04 notebooks)')
print(f'  Window      : {START_DATE} → {END_DATE}')
print(f'  Fill seed   : Jan + Feb 2025 (trimmed after fill)')
print(f'  Split       : {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)}')
print(f'  NaN tolerance: {NAN_TOLERANCE*100:.0f}% (0% = any NaN drops ASIN)')

## ③ Load

In [ ]:
df = pd.read_parquet(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

print(f'Shape        : {df.shape}')
print(f'Unique ASINs : {df["ASIN"].nunique():,}')
print(f'Date range   : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Gender (ASIN level):')
print(df.drop_duplicates('ASIN')['gender'].value_counts().to_string())

## ④ Remove Dirty ASINs — Wrong Gender, Wrong Size, 100% NaN

In [ ]:
# ── Wrong gender / wrong size ─────────────────────────────────────────────
BAD_GENDERS = ['Costumes & Accessories', 'Shoe, Jewelry & Watch Accessories']
gender_size_asins = df[
    df['gender'].isin(BAD_GENDERS) | (df['size'] != '8')
]['ASIN'].unique().tolist()

print(f'Wrong gender/size ASINs ({len(gender_size_asins)}):')
for asin in sorted(gender_size_asins):
    g = df[df['ASIN']==asin]['gender'].iloc[0]
    s = df[df['ASIN']==asin]['size'].iloc[0]
    print(f'  {asin}  gender={g}  size={s}')
log_removal(gender_size_asins, 'wrong gender or size')

# ── 100% NaN in any critical column (checked on Women only) ───────────────
CRITICAL_COLS = ['SALES_RANK', 'PRICE', 'RATING', 'REVIEW_COUNT']
df_women_temp = df[
    (df['gender'] == 'Women') & (~df['ASIN'].isin(gender_size_asins))
].copy()

print(f'\nChecking {df_women_temp["ASIN"].nunique():,} Women ASINs for 100% NaN...')
all_nan_asins = set()
for col in CRITICAL_COLS:
    fully_nan = df_women_temp.groupby('ASIN')[col].apply(lambda x: x.isna().all())
    flagged   = fully_nan[fully_nan].index.tolist()
    if flagged:
        print(f'  {col} — {len(flagged)} ASIN(s) 100% NaN:')
        for asin in sorted(flagged):
            subcat = df_women_temp[df_women_temp['ASIN']==asin]['subcat'].iloc[0]
            print(f'    {asin}  subcat={subcat}')
        all_nan_asins.update(flagged)
    else:
        print(f'  {col} — no 100% NaN ASINs ✅')
log_removal(list(all_nan_asins), '100% NaN in critical column')

dirty_asins = list(set(gender_size_asins) | all_nan_asins)
print(f'\nTotal dirty ASINs to remove: {len(dirty_asins)}')

## ⑤ Filter to Women Only

In [ ]:
before = df['ASIN'].nunique()
df = df[(df['gender'] == 'Women') & (~df['ASIN'].isin(dirty_asins))].copy()
print(f'ASINs : {before:,} → {df["ASIN"].nunique():,}  (kept Women, removed dirty)')
print(f'Rows  : {len(df):,}')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Window distribution:')
print(df['window'].value_counts().to_string())

## ⑥ Drop Dead Columns — Early to Reduce Memory

In [ ]:
cols_dropped = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=cols_dropped)
print(f'Dropped {len(cols_dropped)} columns: {cols_dropped}')
print(f'Remaining ({len(df.columns)}): {df.columns.tolist()}')

## ⑦ Fix Dtypes

In [ ]:
df['window'] = df['window'].astype('float64')
df['date']   = pd.to_datetime(df['date'])

# FBA and FBM — fill NaN with 0
# Keepa did not return offer data = treat as 0 active offers that week
for col in ['Count of retrieved live offers: New, FBA',
            'Count of retrieved live offers: New, FBM']:
    if col in df.columns:
        n = df[col].isna().sum()
        df[col] = df[col].fillna(0).astype('int64')
        short = col.split(': ')[-1]
        print(f'  {short:<8} filled {n:,} NaN → 0  (no Keepa offer data = 0 offers)')

df['New Offer Count: Current']       = df['New Offer Count: Current'].fillna(0).astype('int64')
df['Lightning Deals: Upcoming Deal'] = df['Lightning Deals: Upcoming Deal'].astype('int64')
df['Buy Box: Is FBA']                = df['Buy Box: Is FBA'].astype('int64')
print('✅ Dtypes fixed')

## ⑧ Rebuild subcat_aggregated — Top 10 Women Subcats + Other

In [ ]:
TOP_10 = {
    'Pumps', 'Flats', 'Fashion Sneakers', 'Loafers & Slip-Ons',
    'Walking', 'Mules & Clogs', 'Road Running', 'Heeled Sandals',
    'Ballet & Dance', 'Ankle & Bootie',
}

df['subcat_aggregated'] = df['subcat'].apply(lambda x: x if x in TOP_10 else 'Other')

result = (
    df.drop_duplicates('ASIN')
    .groupby('subcat_aggregated')['ASIN'].count()
    .sort_values(ascending=False).reset_index()
    .rename(columns={'ASIN': 'asin_count'})
)
result['pct'] = (result['asin_count'] / result['asin_count'].sum() * 100).round(1)
print(result.to_string(index=False))
print(f'\nTotal: {result["asin_count"].sum():,} ASINs | {result.shape[0]} groups')

## ⑨ Fill Missing Values — On Full Data Including Jan/Feb

**This is the critical step.**
Jan and Feb 2025 act as the seed for the March window.
We fill NOW, before trimming, so March inherits valid values from earlier months.
BUYBOX_PRICE also fills any leading gaps using the first known price in the window.

In [ ]:
df = df.sort_values(['ASIN', 'window', 'date']).reset_index(drop=True)
fill_cols = [c for c in FFILL_COLS if c in df.columns]

print('NaN counts BEFORE fill (full Jan–Mar data):')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

# Fill all columns forward per (ASIN, window)
# BUYBOX_PRICE additionally fills leading NaN using the first known value
# — products that gained a Buy Box during the window had no prior price data
other_cols  = [c for c in fill_cols if c != 'BUYBOX_PRICE']
buybox_cols = ['BUYBOX_PRICE'] if 'BUYBOX_PRICE' in fill_cols else []

if other_cols:
    df[other_cols] = (
        df.groupby(['ASIN', 'window'])[other_cols]
        .transform(lambda s: s.ffill())
    )

if buybox_cols:
    df['BUYBOX_PRICE'] = (
        df.groupby(['ASIN', 'window'])['BUYBOX_PRICE']
        .transform(lambda s: s.ffill().bfill())
    )

print('\nNaN counts AFTER fill:')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

## ⑩ Trim to Analysis Window, Then Create NaN Flags

Trim Jan/Feb NOW — after fill is done.
Create NaN flags on the trimmed window only, so they accurately reflect
which values were originally missing in the March→Mar 2026 period.

In [ ]:
before_rows = len(df)
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)].copy()

print(f'Rows  : {before_rows:,} → {len(df):,}  (removed {before_rows-len(df):,} Jan/Feb rows)')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()

# NaN check on first date — should be near zero now
first_date = df['date'].min()
fw = df[df['date'] == first_date][fill_cols]
print(f'First date NaN check ({first_date.date()}) — should be near zero after fill:')
for col in fill_cols:
    n = fw[col].isna().sum()
    pct = n / len(fw) * 100
    flag = '✅' if pct < 1 else ('🟡' if pct < 10 else '🔴')
    print(f'  {col:<35} : {n:,} / {len(fw):,}  ({pct:.1f}%)  {flag}')

print()
# Create NaN flags on trimmed window
for col in fill_cols:
    if col in df.columns:
        df[f'is_{col}_filled'] = df[col].isna().astype(int)

print('NaN flag columns created on trimmed window:')
for col in fill_cols:
    flag = f'is_{col}_filled'
    n   = df[flag].sum()
    pct = n / len(df) * 100
    print(f'  {flag:<30} : {n:,} rows were NaN ({pct:.2f}%)')

## ⑪ Find Incomplete ASINs — Show List Before Dropping

An ASIN is incomplete if any of its rows in the analysis window still have NaN
after fill. These cannot be fixed — the ASIN had no data to carry forward even
from Jan/Feb.

`NAN_TOLERANCE = 0.0` means strictly zero NaN allowed (any NaN = drop).
Change this in Config if you want to allow a small % of missing rows.

In [ ]:
all_incomplete = {}

print('Checking for ASINs with remaining NaN after fill...')
print()

for col in REQUIRED_COMPLETE:
    # BUYBOX_PRICE: only drop if 100% missing — partial gaps were already filled
    # All other columns: use NAN_TOLERANCE (default 0 = any NaN = drop)
    threshold = 0.999 if col == 'BUYBOX_PRICE' else NAN_TOLERANCE

    nan_per_asin = (
        df[df['window'] == 7.0]
        .groupby('ASIN')[col]
        .apply(lambda x: (x.isna().sum() / len(x)) > threshold)
    )
    affected = nan_per_asin[nan_per_asin].index.tolist()
    all_incomplete[col] = affected

    if not affected:
        print(f'{col:<35} : no incomplete ASINs ✅')
    else:
        print(f'{col:<35} : {len(affected)} ASINs flagged:')
        for asin in sorted(affected)[:20]:
            subcat    = df[df['ASIN']==asin]['subcat'].iloc[0]
            nan_rows  = df[(df['ASIN']==asin) & df[col].isna()].shape[0]
            total_rows = df[df['ASIN']==asin].shape[0]
            pct_nan   = nan_rows / total_rows * 100
            print(f'  {asin}  subcat={subcat:<35}  NaN={nan_rows}/{total_rows} ({pct_nan:.0f}%)')
        if len(affected) > 20:
            print(f'  ... and {len(affected)-20} more')
    print()

asins_to_drop = set()
for asins in all_incomplete.values():
    asins_to_drop.update(asins)

print(f'Total unique ASINs to drop: {len(asins_to_drop):,}')
pct_lost = len(asins_to_drop) / df['ASIN'].nunique() * 100
print(f'That is {pct_lost:.1f}% of the dataset')
print()
if pct_lost > 10:
    print('⚠️  More than 10% of ASINs would be dropped — review the list above.')
else:
    print('✅ Within acceptable range — safe to drop.')

## ⑫ Execute Drop

In [ ]:
for col, asins in all_incomplete.items():
    if asins:
        log_removal(asins, f'incomplete time series — NaN in {col} after ffill')

before_asins = df['ASIN'].nunique()
before_rows  = len(df)
df = df[~df['ASIN'].isin(asins_to_drop)].copy()

print(f'ASINs : {before_asins:,} → {df["ASIN"].nunique():,}  (dropped {before_asins - df["ASIN"].nunique():,})')
print(f'Rows  : {before_rows:,} → {len(df):,}  (dropped {before_rows - len(df):,})')
print()
print('NaN check after drop:')
for col in REQUIRED_COMPLETE:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  {"✅" if n==0 else "⚠️ "}')

## ⑬ Reorder Columns — Paper Schema

In [ ]:
CORE_COLS = [
    'ASIN', 'window', 'date',
    'SALES_RANK', 'is_SALES_RANK_filled',
    'PRICE', 'is_PRICE_filled',
    'BUYBOX_PRICE', 'is_BUYBOX_PRICE_filled',
    'text',
    'RATING', 'is_RATING_filled',
    'REVIEW_COUNT', 'is_REVIEW_COUNT_filled',
    'subcat', 'subcat_aggregated',
    'New Offer Count: Current',
    'Count of retrieved live offers: New, FBA',
    'Count of retrieved live offers: New, FBM',
    'Lightning Deals: Upcoming Deal',
    'Buy Box: Is FBA',
    'image',
]

missing = [c for c in CORE_COLS if c not in df.columns]
if missing:
    print(f'⚠️  Missing: {missing}')
else:
    df = df[CORE_COLS]
    print(f'✅ {len(df.columns)} columns in correct order')
    for i, col in enumerate(df.columns):
        print(f'  [{i:02d}] {col}')

## ⑭ Schema Verification

In [ ]:
print(f'{"#":<4} {"Column":<50} {"Expected":>16}  {"Actual":>16}')
print('-' * 94)
all_ok = True
for i, col in enumerate(df.columns):
    expected = PAPER_SCHEMA.get(col, 'object')
    actual   = str(df[col].dtype)
    # flag columns are int64 — expected is not in PAPER_SCHEMA, treat as OK
    ok = expected in actual or actual in expected or col.startswith('is_')
    mark = '✅' if ok else '❌'
    if not ok: all_ok = False
    print(f'{i:<4} {col:<50} {expected:>16}  {actual:>16}  {mark}')
print()
print('✅ All dtypes match!' if all_ok else '❌ Some dtypes do not match.')

## ⑮ NaN Final Check

In [ ]:
print(f'Shape : {df.shape}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()
nan_s = df.isna().sum().reset_index()
nan_s.columns = ['column', 'nan_count']
nan_s['nan_pct'] = (nan_s['nan_count'] / len(df) * 100).round(2)
nan_s['status']  = nan_s['nan_pct'].apply(
    lambda x: '🔴 HIGH' if x > 50 else ('🟡 MED' if x > 15 else '✅ OK')
)
print(nan_s.to_string(index=False))
remaining = nan_s[nan_s['nan_count'] > 0]
if len(remaining) == 0:
    print('\n✅ Zero NaN — all columns complete.')
else:
    print(f'\n⚠️  Remaining NaN in {len(remaining)} column(s) — review above.')

## ⑯ Stratified 50/50 ASIN-Level Split

In [ ]:
rng = random.Random(SEED)
train_asins, val_asins = [], []
asin_subcat = df[['ASIN','subcat_aggregated']].drop_duplicates('ASIN')

print(f'Stratified {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} split  seed={SEED}')
print(f'{"subcat_aggregated":<35} {"Total":>6}  {"Train":>8}  {"Val":>6}  {"Train%":>8}')
print('-' * 68)

for subcat, group in asin_subcat.groupby('subcat_aggregated'):
    asins   = sorted(group['ASIN'].unique().tolist())
    rng.shuffle(asins)
    n_train = max(1, round(len(asins) * TRAIN_RATIO))
    train_asins.extend(asins[:n_train])
    val_asins.extend(asins[n_train:])
    pct = n_train / len(asins) * 100
    print(f'  {subcat:<33} {len(asins):>6}  {n_train:>8}  {len(asins)-n_train:>6}  {pct:>7.1f}%')

print('-' * 68)
total = len(train_asins) + len(val_asins)
print(f'  {"TOTAL":<33} {total:>6}  {len(train_asins):>8}  {len(val_asins):>6}  {len(train_asins)/total*100:>7.1f}%')

overlap = set(train_asins) & set(val_asins)
assert len(overlap) == 0, f'ERROR: {len(overlap)} ASINs in both splits!'
print(f'\n✅ No overlap')

## ⑰ Create Train and Val DataFrames

In [ ]:
df_train = df[df['ASIN'].isin(set(train_asins))].copy()
df_val   = df[df['ASIN'].isin(set(val_asins))].copy()

print(f'Train : {len(df_train):,} rows  |  {df_train["ASIN"].nunique():,} ASINs')
print(f'Val   : {len(df_val):,} rows  |  {df_val["ASIN"].nunique():,} ASINs')
print(f'Same date range in both: {df_train["date"].min().date()} → {df_train["date"].max().date()}')

## ⑱ Drop NaN Flag Columns Before Saving

In [ ]:
flag_cols = [f'is_{c}_filled' for c in FFILL_COLS if f'is_{c}_filled' in df.columns]
df_train = df_train.drop(columns=flag_cols)
df_val   = df_val.drop(columns=flag_cols)
print(f'Dropped {len(flag_cols)} flag columns')
print(f'Final columns ({len(df_train.columns)}):')
for i, col in enumerate(df_train.columns):
    print(f'  [{i:02d}] {col:<50} {str(df_train[col].dtype)}')

## ⑲ Save Parquets and Key CSV Files

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(KEY_DIR,  exist_ok=True)

# Save directly to output directories (no Colab temp staging needed)
df_train.to_parquet(DATA_DIR + 'train-00000-of-00001.parquet', index=False)
df_val.to_parquet(DATA_DIR + 'validation-00000-of-00001.parquet', index=False)
print(f'Parquets saved:')
print(f'   train      : {len(df_train):,} rows')
print(f'   validation : {len(df_val):,} rows')

df_train[['ASIN','date']].assign(
    date=df_train['date'].dt.strftime('%Y-%m-%d')
).to_csv(KEY_DIR + 'main_train_keys.csv', index=False)
df_val[['ASIN','date']].assign(
    date=df_val['date'].dt.strftime('%Y-%m-%d')
).to_csv(KEY_DIR + 'main_val_keys.csv', index=False)
print(f'Key CSV files saved to:')
print(f'   {KEY_DIR}main_train_keys.csv')
print(f'   {KEY_DIR}main_val_keys.csv')
print()
print('These sit in the code/ folder next to 01_1, 01_2, and 04 notebooks.')

## ⑳ Complete ASIN Removal Log

In [ ]:
log_df = pd.DataFrame(removal_log)

print('=' * 65)
print('COMPLETE ASIN REMOVAL LOG')
print('=' * 65)
print(f'Total ASINs removed : {log_df["ASIN"].nunique():,}')
print(f'Total ASINs kept    : {df["ASIN"].nunique():,}')
print(f'Started with        : {log_df["ASIN"].nunique() + df["ASIN"].nunique():,} Women ASINs')
print()

# Summary by reason
summary = log_df.groupby('reason')['ASIN'].nunique().reset_index()
summary.columns = ['reason', 'count']
print('By reason:')
print(summary.to_string(index=False))
print()

# Detailed list by reason
for reason, group in log_df.groupby('reason'):
    asins = sorted(group['ASIN'].unique())
    print(f'── {reason}  ({len(asins)} ASINs) ──')
    for asin in asins:
        # Try to find subcat from the full dataset context
        rows = df_train[df_train['ASIN']==asin]
        if len(rows) == 0:
            rows = df_val[df_val['ASIN']==asin]
        subcat = rows['subcat'].iloc[0] if len(rows) > 0 else 'n/a (removed before split)'
        print(f'  {asin}  {subcat}')
    print()

## ㉑ Final Verification

In [ ]:
df_tr = pd.read_parquet(DATA_DIR + 'train-00000-of-00001.parquet')
df_vl = pd.read_parquet(DATA_DIR + 'validation-00000-of-00001.parquet')

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Train  : {len(df_tr):,} rows  |  {df_tr["ASIN"].nunique():,} ASINs')
print(f'Val    : {len(df_vl):,} rows  |  {df_vl["ASIN"].nunique():,} ASINs')
print(f'Columns: {len(df_tr.columns)}')
print(f'Date   : {df_tr["date"].min()} → {df_tr["date"].max()}')
print()

overlap = set(df_tr['ASIN'].unique()) & set(df_vl['ASIN'].unique())
print(f'ASIN overlap : {len(overlap)}  {"✅" if len(overlap)==0 else "❌"}')
print()

print('subcat balance:')
tr_c = df_tr.drop_duplicates('ASIN')['subcat_aggregated'].value_counts().rename('train')
vl_c = df_vl.drop_duplicates('ASIN')['subcat_aggregated'].value_counts().rename('val')
bal  = pd.concat([tr_c, vl_c], axis=1).fillna(0).astype(int)
bal['total']   = bal['train'] + bal['val']
bal['train_%'] = (bal['train'] / bal['total'] * 100).round(1)
print(bal.sort_values('total', ascending=False).to_string())
print()

print('NaN check on saved files:')
all_clean = True
for col in ['SALES_RANK','PRICE','BUYBOX_PRICE','RATING','REVIEW_COUNT']:
    n_tr = df_tr[col].isna().sum()
    n_vl = df_vl[col].isna().sum()
    flag = '✅' if n_tr==0 and n_vl==0 else '⚠️ '
    if n_tr > 0 or n_vl > 0: all_clean = False
    print(f'  {col:<35} train={n_tr:,}  val={n_vl:,}  {flag}')

print()
if all_clean:
    print('✅ Data preparation complete — all columns clean!')
else:
    print('⚠️  Some NaN remain — review removal log and consider adjusting NAN_TOLERANCE')
print(f'\nNext: run image download notebook (~{df_tr["ASIN"].nunique()+df_vl["ASIN"].nunique():,} images)')